# Experiment 14 — Row-Level Mutations (PySpark companion notebook)

This notebook mirrors [`experiments/14-row-level-mutations.md`](../../../../experiments/14-row-level-mutations.md). The markdown is the canonical writeup — read its **interview question**, **break it**, and **theory deep-dive** sections in order. This notebook only carries the hands-on cells so you can re-run them in PySpark style without leaving Jupyter.

**Where to run this**: inside the `spark-iceberg` container in `lab/spark-profile/`.

```bash
cd lab/spark-profile
./up.sh
docker compose exec spark-iceberg pyspark --conf spark.sql.shuffle.partitions=4
```

Or launch a notebook server inside the container if you prefer. The Spark session is already configured to use the `rest_lab` Iceberg REST catalog backed by Lakekeeper + MinIO.

## Step 0 — Get a Spark session

If you launched via `pyspark` the variable `spark` is already bound. The cell below is idempotent: if a session exists it just re-attaches.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sql("SHOW CURRENT NAMESPACE").show()
spark.sql("SHOW CATALOGS").show()

## Step 1 — Create a copy-on-write table

Iceberg V2 with all three mutation modes set to `copy-on-write`. Partitioned by `status` so we can see partition-scoped rewrites.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo")
spark.sql("DROP TABLE IF EXISTS demo.orders_cow")
spark.sql("""
CREATE TABLE demo.orders_cow (
    order_id   BIGINT,
    user_id    BIGINT,
    amount     DOUBLE,
    status     STRING,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (status)
TBLPROPERTIES (
    'format-version'    = '2',
    'write.delete.mode' = 'copy-on-write',
    'write.update.mode' = 'copy-on-write',
    'write.merge.mode'  = 'copy-on-write'
)
""")

spark.sql("""
INSERT INTO demo.orders_cow VALUES
    (1, 100, 49.99, 'pending',   TIMESTAMP '2026-05-01 09:00:00'),
    (2, 101, 19.99, 'pending',   TIMESTAMP '2026-05-01 09:01:00'),
    (3, 102, 99.50, 'pending',   TIMESTAMP '2026-05-01 09:02:00'),
    (4, 100, 14.99, 'completed', TIMESTAMP '2026-05-01 09:03:00'),
    (5, 103,  4.99, 'completed', TIMESTAMP '2026-05-01 09:04:00')
""")

spark.sql("SELECT file_path, file_format, record_count, file_size_in_bytes FROM demo.orders_cow.files").show(truncate=False)

**Observe**: two data files, one per partition value (`status=pending`, `status=completed`).

Snapshot the pre-delete state before continuing — we'll time-travel back to it as a fresh seed for the MoR table.

In [ ]:
snap_before = (
    spark.sql("SELECT snapshot_id FROM demo.orders_cow.snapshots ORDER BY committed_at DESC LIMIT 1")
    .collect()[0][0]
)
print("SNAP_BEFORE =", snap_before)

## Step 2 — Delete one row in CoW mode

We delete a single row out of the `pending` partition. CoW will rewrite the partition's data file.

In [ ]:
spark.sql("DELETE FROM demo.orders_cow WHERE order_id = 2")

spark.sql("SELECT content, file_path, record_count FROM demo.orders_cow.files ORDER BY file_path").show(truncate=False)

Notice:
- The `pending` partition has a **new file path** with **2 records** instead of 3.
- The `completed` partition file is **unchanged**.
- No `content > 0` rows appear — CoW never writes delete files.

Verify time travel still sees the deleted row:

In [ ]:
spark.sql(f"SELECT order_id, status FROM demo.orders_cow VERSION AS OF {snap_before} ORDER BY order_id").show()

## Step 3 — Create a merge-on-read table

Seed it from the *pre-delete* version of `orders_cow` so it starts with all 5 rows.

In [ ]:
spark.sql("DROP TABLE IF EXISTS demo.orders_mor")
spark.sql("""
CREATE TABLE demo.orders_mor (
    order_id   BIGINT,
    user_id    BIGINT,
    amount     DOUBLE,
    status     STRING,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (status)
TBLPROPERTIES (
    'format-version'    = '2',
    'write.delete.mode' = 'merge-on-read',
    'write.update.mode' = 'merge-on-read',
    'write.merge.mode'  = 'merge-on-read'
)
""")

spark.sql(f"INSERT INTO demo.orders_mor SELECT * FROM demo.orders_cow VERSION AS OF {snap_before}")
spark.sql("SELECT * FROM demo.orders_mor ORDER BY order_id").show()

## Step 4 — Delete one row in MoR mode

Same logical operation, completely different physical effect: a tiny **position delete file** instead of a data file rewrite.

In [ ]:
spark.sql("DELETE FROM demo.orders_mor WHERE order_id = 2")

spark.sql("""
SELECT content, file_path, record_count, file_size_in_bytes
FROM demo.orders_mor.files
ORDER BY content, file_path
""").show(truncate=False)

`content` legend:
- `0` — data file (Parquet)
- `1` — position delete file (Parquet listing `(referenced_data_file, position)` tuples)
- `2` — equality delete file (Parquet listing column values that match dead rows)

Inspect what the delete file actually points at:

In [ ]:
spark.sql("""
SELECT data_file.content,
       data_file.file_path,
       data_file.referenced_data_file,
       data_file.record_count
FROM demo.orders_mor.entries
WHERE data_file.content >= 1
""").show(truncate=False)

## Step 5 — Pay the read cost, then compact it away

Every read of `orders_mor` now joins the data files against the position-delete file. Let's see it, then collapse it back into a single data file per partition with `rewrite_data_files`.

In [ ]:
spark.sql("SELECT * FROM demo.orders_mor ORDER BY order_id").show()

In [ ]:
spark.sql("""
CALL rest_lab.system.rewrite_data_files(
    table   => 'demo.orders_mor',
    options => map('delete-file-threshold', '1')
)
""").show(truncate=False)

spark.sql("""
SELECT content, COUNT(*) AS files, SUM(record_count) AS rows
FROM demo.orders_mor.files
GROUP BY content
ORDER BY content
""").show()

All `content` rows should now be `0` — delete files have been materialized into rewritten data files.

## Step 6 — `MERGE INTO` (the CDC bread-and-butter)

Simulate a CDC batch that updates two rows, deletes one (via the `cancelled` status), and inserts two new ones.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW cdc_batch AS
SELECT * FROM VALUES
    (1, 100, 49.99, 'completed', TIMESTAMP '2026-05-01 10:00:00'),
    (3, 102, 99.50, 'cancelled', TIMESTAMP '2026-05-01 10:01:00'),
    (6, 104, 29.00, 'pending',   TIMESTAMP '2026-05-01 10:02:00'),
    (7, 105,  9.99, 'pending',   TIMESTAMP '2026-05-01 10:03:00')
AS t(order_id, user_id, amount, status, updated_at)
""")

spark.sql("""
MERGE INTO demo.orders_mor t
USING cdc_batch s
ON t.order_id = s.order_id
WHEN MATCHED AND s.status = 'cancelled' THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("SELECT * FROM demo.orders_mor ORDER BY order_id").show()

In [ ]:
spark.sql("""
SELECT data_file.content,
       data_file.equality_ids,
       data_file.record_count,
       data_file.file_path
FROM demo.orders_mor.entries
ORDER BY data_file.content DESC
""").show(truncate=False)

Observe how the MERGE manifested physically:
- New data files (`content=0`) holding inserted + updated rows
- An **equality delete file** (`content=2`, `equality_ids = [1]` if `order_id` is field-id 1) listing the keys whose old rows are dead

## Step 7 — Compare CoW vs MoR side by side

Apply the same MERGE batch against the CoW table, then look at the file footprint of each.

In [ ]:
spark.sql("""
MERGE INTO demo.orders_cow t
USING cdc_batch s
ON t.order_id = s.order_id
WHEN MATCHED AND s.status = 'cancelled' THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("""
SELECT 'cow' AS mode,
       COUNT(*) FILTER (WHERE content = 0) AS data_files,
       COUNT(*) FILTER (WHERE content = 1) AS pos_deletes,
       COUNT(*) FILTER (WHERE content = 2) AS eq_deletes,
       SUM(file_size_in_bytes)            AS total_bytes
FROM demo.orders_cow.files
UNION ALL
SELECT 'mor',
       COUNT(*) FILTER (WHERE content = 0),
       COUNT(*) FILTER (WHERE content = 1),
       COUNT(*) FILTER (WHERE content = 2),
       SUM(file_size_in_bytes)
FROM demo.orders_mor.files
""").show()

**The single takeaway**: CoW spends more bytes once (the rewrite). MoR spends fewer bytes but keeps spending — every read has to apply the deletes, until compaction runs.

## Break it

Run these only after you've understood the happy path. See the markdown experiment file for the full break-it discussion.

### Break 1 — kill a delete file in MinIO

Manual step: open MinIO at http://localhost:9001, find a file matching `*-deletes-*.parquet` under `demo/orders_mor/`, delete it from the console. Then re-run:

In [ ]:
spark.sql("SELECT order_id, status FROM demo.orders_mor ORDER BY order_id").show()

Previously-deleted rows reappear. Delete files are load-bearing; lose one and you've changed the answer the table returns.

Roll back to the snapshot before you broke it:

In [ ]:
snaps = spark.sql("SELECT snapshot_id, committed_at FROM demo.orders_mor.snapshots ORDER BY committed_at DESC").collect()
for row in snaps:
    print(row.snapshot_id, row.committed_at)
# Then, picking a known-good snapshot id:
# spark.sql("CALL rest_lab.system.rollback_to_snapshot('demo.orders_mor', <id>)")

### Break 2 — read MoR from DuckDB in the main lab

This cell is meant to be run from the **main lab's Jupyter** (not this Spark container) to exercise a non-Spark reader against the V2 MoR table you just wrote. From the main lab's Jupyter:

```python
import duckdb
from src.catalog_helper import configure_duckdb_for_minio
con = duckdb.connect()
configure_duckdb_for_minio(con)
con.execute("SELECT * FROM iceberg_scan('s3://warehouse/demo/orders_mor/')").fetchdf()
```

Depending on DuckDB's iceberg extension version you may get the correct answer, a wrong answer that ignores equality deletes, or an error. The lesson is that V2 MoR features land in non-Spark engines on a long tail — assume nothing until you've tested every reader.

### Break 3 — pile up delete files

Loop many single-row deletes and watch delete-file count climb.

In [ ]:
# Inflate the table first
spark.sql("""
INSERT INTO demo.orders_mor
SELECT order_id + 1000, user_id, amount, status, updated_at
FROM demo.orders_mor
""")

for oid in range(1001, 1011):
    spark.sql(f"DELETE FROM demo.orders_mor WHERE order_id = {oid}")

spark.sql("""
SELECT content, COUNT(*) AS files
FROM demo.orders_mor.files
GROUP BY content
ORDER BY content
""").show()

In [ ]:
import time
t0 = time.time()
spark.sql("SELECT COUNT(*) FROM demo.orders_mor").show()
print(f"Scan latency before compaction: {(time.time()-t0)*1000:.0f} ms")

spark.sql("""
CALL rest_lab.system.rewrite_data_files(
    table   => 'demo.orders_mor',
    options => map('delete-file-threshold', '1')
)
""").show(truncate=False)

t0 = time.time()
spark.sql("SELECT COUNT(*) FROM demo.orders_mor").show()
print(f"Scan latency after compaction:  {(time.time()-t0)*1000:.0f} ms")

### Break 4 — MERGE with a non-unique join key

Iceberg via Spark enforces target-row uniqueness in MERGE. A duplicate-keyed source raises rather than corrupting silently.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW bad_source AS
SELECT 1 AS order_id, 100 AS user_id, 49.99 AS amount, 'completed' AS status, TIMESTAMP '2026-05-01' AS updated_at
UNION ALL
SELECT 1, 100, 49.99, 'cancelled', TIMESTAMP '2026-05-01'
""")

try:
    spark.sql("""
    MERGE INTO demo.orders_cow t USING bad_source s
    ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET *
    """)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

## Session wrap-up

1. Re-answer the interview question in `interview-faq.md` — physical model of CoW vs MoR, position vs equality deletes, MERGE semantics, engine-compat caveat.
2. To wipe state for a clean re-run: roll back to an early snapshot, or `cd lab && ./reset.sh --confirm` to drop everything.
3. Stop the Spark profile when done: `cd lab/spark-profile && ./down.sh`. The main lab can stay running for other experiments.